# 从零复现 Faster R-CNN：anchors、RPN、近似 ROI Pooling 与两阶段检测

本 Notebook 不调用 `torchvision` 检测器、`torchvision.ops`、预训练 backbone 或现成 NMS/ROI Align。我们手写 Tiny backbone、anchor 生成、IoU 匹配与均衡采样、box delta 编解码、RPN、proposal 过滤、NMS、整数边界 ROI pooling、二阶段分类/回归头、训练损失和推理后处理。

这里的类名明确使用 `IntegerROIPool`，因为它采用整数裁剪加 adaptive max pooling，**不是**带双线性采样的 ROI Align。全部离线、CPU 单线程；小方块任务只验证两阶段数据流和梯度，不追求或声称真实 mAP。


## 1. 两阶段计算图与坐标合同

```text
images + padding_mask
  -> TinyBackbone -> feature + conservative feature padding mask
  -> RPN conv -> objectness [B,A*Hf*Wf] + deltas [B,A*Hf*Wf,4]
  -> decode -> clip -> size filter -> top-k -> NMS -> proposals
  -> IntegerROIPool(feature, proposals) -> [R,C,3,3]
  -> TwoStageHead -> class logits [R,K+1] + class-agnostic deltas [R,4]
  -> train: RPN loss + ROI classification/regression loss
  -> infer: decode -> per-class NMS -> detections
```

本册统一采用连续边界 `xyxy=(x1,y1,x2,y2)`，范围为 `0<=x<=W, 0<=y<=H`，宽高分别是 `x2-x1,y2-y1`；类别 `0` 专用于 background，真实类别从 1 开始。把 inclusive pixel 坐标与连续边界混用，会产生系统性的 `+1/-1` 误差。


In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from copy import deepcopy  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 470728  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.use_deterministic_algorithms(True)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。


## 2. Box、IoU 与 delta：先锁住几何语义

给 anchor $a=(x_a,y_a,w_a,h_a)$ 和目标 $g$，常用参数化是

$$t_x=(x_g-x_a)/w_a,\;t_y=(y_g-y_a)/h_a,\;t_w=\log(w_g/w_a),\;t_h=\log(h_g/h_a).$$

decode 是它的逆变换。宽高的指数必须 clamp，否则异常 delta 会溢出。下面不仅测 shape，还验证 encode/decode 往返和手算 IoU；退化 box、NaN、越界 GT 都会 fail closed。


In [ ]:
def validate_boxes(boxes, image_size=None, allow_empty=True):  # 定义本节可复用的核心函数。
    if boxes.ndim != 2 or boxes.shape[-1] != 4 or not torch.is_floating_point(boxes):  # 按当前条件选择后续控制路径。
        raise ValueError("boxes must be floating [N,4]")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(boxes).all():  # 按当前条件选择后续控制路径。
        raise ValueError("boxes must be finite")  # 遇到非法合同立即显式失败。
    if boxes.numel() == 0:  # 按当前条件选择后续控制路径。
        if allow_empty:  # 按当前条件选择后续控制路径。
            return  # 返回当前分支计算出的结果。
        raise ValueError("boxes cannot be empty")  # 遇到非法合同立即显式失败。
    if not ((boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])).all():  # 按当前条件选择后续控制路径。
        raise ValueError("boxes must have positive extent")  # 遇到非法合同立即显式失败。
    if image_size is not None:  # 按当前条件选择后续控制路径。
        h, w = image_size  # 计算并保存当前步骤的中间状态。
        if (boxes[:, 0] < 0).any() or (boxes[:, 1] < 0).any() or (boxes[:, 2] > w).any() or (boxes[:, 3] > h).any():  # 按当前条件选择后续控制路径。
            raise ValueError("boxes exceed continuous image bounds")  # 遇到非法合同立即显式失败。

def box_iou(boxes1, boxes2):  # 定义本节可复用的核心函数。
    validate_boxes(boxes1)  # 执行当前语句以推进本节示例。
    validate_boxes(boxes2)  # 执行当前语句以推进本节示例。
    if boxes1.shape[0] == 0 or boxes2.shape[0] == 0:  # 按当前条件选择后续控制路径。
        return boxes1.new_zeros((boxes1.shape[0], boxes2.shape[0]))  # 返回当前分支计算出的结果。
    top_left = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])  # 计算并保存当前步骤的中间状态。
    bottom_right = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])  # 计算并保存当前步骤的中间状态。
    intersection = (bottom_right - top_left).clamp(min=0).prod(-1)  # 计算并保存当前步骤的中间状态。
    area1 = (boxes1[:, 2:] - boxes1[:, :2]).prod(-1)  # 计算并保存当前步骤的中间状态。
    area2 = (boxes2[:, 2:] - boxes2[:, :2]).prod(-1)  # 计算并保存当前步骤的中间状态。
    return intersection / (area1[:, None] + area2[None, :] - intersection).clamp(min=1e-8)  # 返回当前分支计算出的结果。

class BoxCoder(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, max_log_scale=math.log(1000.0 / 16)):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.max_log_scale = float(max_log_scale)  # 计算并保存当前步骤的中间状态。

    def encode(self, anchors, targets):  # 定义本节可复用的核心函数。
        validate_boxes(anchors); validate_boxes(targets)  # 执行当前语句以推进本节示例。
        if anchors.shape != targets.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("anchors and targets must align")  # 遇到非法合同立即显式失败。
        ac = (anchors[:, :2] + anchors[:, 2:]) / 2  # 计算并保存当前步骤的中间状态。
        awh = anchors[:, 2:] - anchors[:, :2]  # 计算并保存当前步骤的中间状态。
        gc = (targets[:, :2] + targets[:, 2:]) / 2  # 计算并保存当前步骤的中间状态。
        gwh = targets[:, 2:] - targets[:, :2]  # 计算并保存当前步骤的中间状态。
        return torch.cat([(gc - ac) / awh, torch.log(gwh / awh)], dim=-1)  # 返回当前分支计算出的结果。

    def decode(self, anchors, deltas):  # 定义本节可复用的核心函数。
        validate_boxes(anchors)  # 执行当前语句以推进本节示例。
        if deltas.shape != anchors.shape or not torch.isfinite(deltas).all():  # 按当前条件选择后续控制路径。
            raise ValueError("deltas must be finite and align with anchors")  # 遇到非法合同立即显式失败。
        ac = (anchors[:, :2] + anchors[:, 2:]) / 2  # 计算并保存当前步骤的中间状态。
        awh = anchors[:, 2:] - anchors[:, :2]  # 计算并保存当前步骤的中间状态。
        center = ac + deltas[:, :2] * awh  # 计算并保存当前步骤的中间状态。
        wh = awh * deltas[:, 2:].clamp(max=self.max_log_scale).exp()  # 计算并保存当前步骤的中间状态。
        return torch.cat([center - wh / 2, center + wh / 2], dim=-1)  # 返回当前分支计算出的结果。

    def forward(self, anchors, deltas):  # 定义本节可复用的核心函数。
        return self.decode(anchors, deltas)  # 返回当前分支计算出的结果。

def clip_boxes(boxes, image_size):  # 定义本节可复用的核心函数。
    if boxes.ndim != 2 or boxes.shape[-1] != 4 or not torch.isfinite(boxes).all():  # 按当前条件选择后续控制路径。
        raise ValueError("finite [N,4] boxes required")  # 遇到非法合同立即显式失败。
    h, w = image_size  # 计算并保存当前步骤的中间状态。
    return torch.stack([boxes[:, 0].clamp(0, w), boxes[:, 1].clamp(0, h),  # 返回当前分支计算出的结果。
                        boxes[:, 2].clamp(0, w), boxes[:, 3].clamp(0, h)], dim=-1)  # 计算并保存当前步骤的中间状态。

box_coder47 = BoxCoder()  # 计算并保存当前步骤的中间状态。
anchors_oracle = torch.tensor([[0., 0., 10., 10.], [10., 8., 20., 24.]])  # 计算并保存当前步骤的中间状态。
targets_oracle = torch.tensor([[1., 2., 9., 8.], [8., 10., 24., 22.]])  # 计算并保存当前步骤的中间状态。
deltas_oracle = box_coder47.encode(anchors_oracle, targets_oracle)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(box_coder47.decode(anchors_oracle, deltas_oracle), targets_oracle, atol=1e-5)  # 用受控断言验证关键不变量。
iou_oracle = box_iou(torch.tensor([[0., 0., 2., 2.]]), torch.tensor([[1., 1., 3., 3.]]))  # 计算并保存当前步骤的中间状态。
assert torch.allclose(iou_oracle, torch.tensor([[1 / 7]]), atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.equal(clip_boxes(torch.tensor([[-2., -1., 35., 40.]]), (32, 32)), torch.tensor([[0., 0., 32., 32.]]))  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    validate_boxes(torch.tensor([[1., 1., 1., 3.]]))  # 执行当前语句以推进本节示例。
    raise AssertionError("degenerate boxes must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。


## 3. Tiny backbone 与 padding 感受野

仅把输入 padding 像素设零还不够：卷积边界附近的 feature cell 感受野可能跨入 padding。这里逐层用与卷积相同的 kernel/stride/padding 对 invalid mask 做 max pooling；任一输入无效，该 feature cell 就标为无效并清零。这是保守策略，会牺牲少量边界特征，但保证调用方任意修改 padding 值都不影响模型输出。


In [ ]:
class TinyDetectionBackbone(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, out_channels=16):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.in_channels = int(in_channels)  # 计算并保存当前步骤的中间状态。
        self.out_channels = int(out_channels)  # 计算并保存当前步骤的中间状态。
        self.conv1 = nn.Conv2d(in_channels, 12, 3, stride=2, padding=1)  # 计算并保存当前步骤的中间状态。
        self.conv2 = nn.Conv2d(12, out_channels, 3, stride=2, padding=1)  # 计算并保存当前步骤的中间状态。

    @staticmethod  # 为下方定义附加声明式配置。
    def _advance_mask(mask):  # 定义本节可复用的核心函数。
        return F.max_pool2d(mask.float().unsqueeze(1), 3, stride=2, padding=1).squeeze(1).bool()  # 返回当前分支计算出的结果。

    def forward(self, images, padding_mask):  # 定义本节可复用的核心函数。
        if images.ndim != 4 or images.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("images must be configured NCHW")  # 遇到非法合同立即显式失败。
        if padding_mask.shape != images.shape[:1] + images.shape[-2:] or padding_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("padding_mask must be bool [B,H,W]")  # 遇到非法合同立即显式失败。
        if not torch.isfinite(images).all():  # 按当前条件选择后续控制路径。
            raise ValueError("images must be finite")  # 遇到非法合同立即显式失败。
        x = images.masked_fill(padding_mask[:, None], 0)  # 计算并保存当前步骤的中间状态。
        mask = self._advance_mask(padding_mask)  # 计算并保存当前步骤的中间状态。
        x = F.relu(self.conv1(x)).masked_fill(mask[:, None], 0)  # 计算并保存当前步骤的中间状态。
        mask = self._advance_mask(mask)  # 计算并保存当前步骤的中间状态。
        x = F.relu(self.conv2(x)).masked_fill(mask[:, None], 0)  # 计算并保存当前步骤的中间状态。
        return x, mask  # 返回当前分支计算出的结果。

backbone47 = TinyDetectionBackbone()  # 计算并保存当前步骤的中间状态。
pad_mask_probe = torch.zeros(1, 32, 32, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
pad_mask_probe[:, 24:, :] = True; pad_mask_probe[:, :, 24:] = True  # 计算并保存当前步骤的中间状态。
pad_a = torch.randn(1, 1, 32, 32)  # 计算并保存当前步骤的中间状态。
pad_b = pad_a.clone(); pad_b[pad_mask_probe[:, None]] = 999.0  # 计算并保存当前步骤的中间状态。
feat_a, fmask_a = backbone47(pad_a, pad_mask_probe)  # 计算并保存当前步骤的中间状态。
feat_b, fmask_b = backbone47(pad_b, pad_mask_probe)  # 计算并保存当前步骤的中间状态。
assert feat_a.shape == (1, 16, 8, 8)  # 用受控断言验证关键不变量。
assert torch.equal(fmask_a, fmask_b)  # 用受控断言验证关键不变量。
assert torch.equal(feat_a, feat_b)  # 用受控断言验证关键不变量。
assert torch.equal(feat_a.masked_select(fmask_a[:, None]), torch.zeros_like(feat_a.masked_select(fmask_a[:, None])))  # 用受控断言验证关键不变量。


## 4. Anchors、IoU 匹配与均衡采样

每个 feature cell 以其中心为 anchor 中心，并枚举三个 aspect ratio。RPN label 为 `1=positive, 0=negative, -1=ignore`：IoU 高于正阈值为正、低于负阈值为负，中间忽略；另外强制每个 GT 的最佳 anchor 为正，避免小物体没有监督。空 GT 图像的所有 anchor 都是负样本，这是目标检测训练必须覆盖的正常情况。


In [ ]:
RPN_POSITIVE_IOU47 = 0.5  # 计算并保存当前步骤的中间状态。
RPN_NEGATIVE_IOU47 = 0.2  # 计算并保存当前步骤的中间状态。
RPN_BATCH_SIZE47 = 64  # 计算并保存当前步骤的中间状态。
RPN_POSITIVE_FRACTION47 = 0.5  # 计算并保存当前步骤的中间状态。

class AnchorGenerator(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, size=8.0, aspect_ratios=(0.5, 1.0, 2.0)):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if size <= 0 or not aspect_ratios or any(r <= 0 for r in aspect_ratios):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid anchor recipe")  # 遇到非法合同立即显式失败。
        self.size = float(size)  # 计算并保存当前步骤的中间状态。
        self.aspect_ratios = tuple(float(r) for r in aspect_ratios)  # 计算并保存当前步骤的中间状态。

    def forward(self, feature_size, image_size):  # 定义本节可复用的核心函数。
        hf, wf = feature_size; hi, wi = image_size  # 计算并保存当前步骤的中间状态。
        if min(hf, wf, hi, wi) <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("feature/image dimensions must be positive")  # 遇到非法合同立即显式失败。
        stride_y, stride_x = hi / hf, wi / wf  # 计算并保存当前步骤的中间状态。
        cy = (torch.arange(hf, dtype=torch.float32) + 0.5) * stride_y  # 计算并保存当前步骤的中间状态。
        cx = (torch.arange(wf, dtype=torch.float32) + 0.5) * stride_x  # 计算并保存当前步骤的中间状态。
        yy, xx = torch.meshgrid(cy, cx, indexing="ij")  # 计算并保存当前步骤的中间状态。
        all_anchors = []  # 计算并保存当前步骤的中间状态。
        for ratio in self.aspect_ratios:  # 遍历输入元素以累积或检查结果。
            width = self.size * math.sqrt(ratio)  # 计算并保存当前步骤的中间状态。
            height = self.size / math.sqrt(ratio)  # 计算并保存当前步骤的中间状态。
            all_anchors.append(torch.stack([xx - width/2, yy - height/2, xx + width/2, yy + height/2], -1))  # 执行当前语句以推进本节示例。
        return torch.stack(all_anchors, 2).reshape(-1, 4)  # 返回当前分支计算出的结果。

def match_anchors(anchors, gt_boxes, positive_iou=RPN_POSITIVE_IOU47, negative_iou=RPN_NEGATIVE_IOU47):  # 定义本节可复用的核心函数。
    validate_boxes(anchors); validate_boxes(gt_boxes)  # 执行当前语句以推进本节示例。
    if not 0 <= negative_iou < positive_iou <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("invalid IoU thresholds")  # 遇到非法合同立即显式失败。
    labels = torch.full((anchors.shape[0],), -1, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    matched = torch.zeros(anchors.shape[0], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    if gt_boxes.shape[0] == 0:  # 按当前条件选择后续控制路径。
        labels.fill_(0)  # 执行当前语句以推进本节示例。
        return labels, matched  # 返回当前分支计算出的结果。
    ious = box_iou(anchors, gt_boxes)  # 计算并保存当前步骤的中间状态。
    best_iou, matched = ious.max(dim=1)  # 计算并保存当前步骤的中间状态。
    labels[best_iou < negative_iou] = 0  # 计算并保存当前步骤的中间状态。
    labels[best_iou >= positive_iou] = 1  # 计算并保存当前步骤的中间状态。
    best_anchor_per_gt = ious.argmax(dim=0)  # 计算并保存当前步骤的中间状态。
    labels[best_anchor_per_gt] = 1  # 计算并保存当前步骤的中间状态。
    matched[best_anchor_per_gt] = torch.arange(gt_boxes.shape[0])  # 计算并保存当前步骤的中间状态。
    return labels, matched  # 返回当前分支计算出的结果。

def balanced_sample(labels, batch_size, positive_fraction, generator):  # 定义本节可复用的核心函数。
    if labels.ndim != 1 or labels.dtype != torch.long or not set(labels.unique().tolist()).issubset({-1, 0, 1}):  # 按当前条件选择后续控制路径。
        raise ValueError("labels must use {-1,0,1}")  # 遇到非法合同立即显式失败。
    if not isinstance(batch_size, int) or batch_size <= 0 or not 0 <= positive_fraction <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("sampling batch_size/fraction contract is invalid")  # 遇到非法合同立即显式失败。
    if not isinstance(generator, torch.Generator):  # 按当前条件选择后续控制路径。
        raise ValueError("balanced sampling requires an explicit torch.Generator")  # 遇到非法合同立即显式失败。
    positive = torch.where(labels == 1)[0]  # 计算并保存当前步骤的中间状态。
    negative = torch.where(labels == 0)[0]  # 计算并保存当前步骤的中间状态。
    npos = min(int(batch_size * positive_fraction), positive.numel())  # 计算并保存当前步骤的中间状态。
    nneg = min(batch_size - npos, negative.numel())  # 计算并保存当前步骤的中间状态。
    pos = positive[torch.randperm(positive.numel(), generator=generator)[:npos]]  # 计算并保存当前步骤的中间状态。
    neg = negative[torch.randperm(negative.numel(), generator=generator)[:nneg]]  # 计算并保存当前步骤的中间状态。
    return torch.cat([pos, neg]), pos  # 返回当前分支计算出的结果。

anchor_generator47 = AnchorGenerator()  # 计算并保存当前步骤的中间状态。
anchors47 = anchor_generator47((8, 8), (32, 32))  # 计算并保存当前步骤的中间状态。
assert anchors47.shape == (8 * 8 * 3, 4)  # 用受控断言验证关键不变量。
empty_labels, empty_match = match_anchors(anchors47, torch.empty(0, 4))  # 计算并保存当前步骤的中间状态。
assert torch.equal(empty_labels, torch.zeros_like(empty_labels))  # 用受控断言验证关键不变量。
assert empty_match.shape == empty_labels.shape  # 用受控断言验证关键不变量。
one_labels, one_match = match_anchors(anchors47, torch.tensor([[8., 8., 16., 16.]]))  # 计算并保存当前步骤的中间状态。
assert bool((one_labels == 1).any()) and bool((one_labels == 0).any()) and bool((one_labels == -1).any())  # 用受控断言验证关键不变量。
sampled47, sampled_pos47 = balanced_sample(one_labels, 32, 0.5, torch.Generator().manual_seed(4))  # 计算并保存当前步骤的中间状态。
sampled_again47, _ = balanced_sample(one_labels, 32, 0.5, torch.Generator().manual_seed(4))  # 计算并保存当前步骤的中间状态。
assert sampled47.numel() <= 32 and sampled_pos47.numel() <= 16  # 用受控断言验证关键不变量。
assert sampled47.unique().numel() == sampled47.numel() and not bool((one_labels[sampled47] == -1).any())  # 用受控断言验证关键不变量。
assert torch.equal(sampled47, sampled_again47)  # 用受控断言验证关键不变量。


## 5. RPN：objectness 与 class-agnostic box regression

共享 `3×3` 卷积后，每个位置、每个 anchor 输出一个 objectness logit 和四个 delta。分类损失只看 sampled positive/negative；回归损失只看 positive。禁止对 ignore anchor 算 BCE，也不能在空 GT 时对空回归张量求 mean（会得到 NaN）。


In [ ]:
RPN_SMOOTH_L1_BETA47 = 1 / 9  # 计算并保存当前步骤的中间状态。

class RegionProposalNetwork(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, anchors_per_location=3):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if in_channels <= 0 or anchors_per_location <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("RPN dimensions must be positive")  # 遇到非法合同立即显式失败。
        self.anchors_per_location = int(anchors_per_location)  # 计算并保存当前步骤的中间状态。
        self.shared = nn.Conv2d(in_channels, in_channels, 3, padding=1)  # 计算并保存当前步骤的中间状态。
        self.objectness = nn.Conv2d(in_channels, anchors_per_location, 1)  # 计算并保存当前步骤的中间状态。
        self.box_deltas = nn.Conv2d(in_channels, 4 * anchors_per_location, 1)  # 计算并保存当前步骤的中间状态。

    def forward(self, features):  # 定义本节可复用的核心函数。
        if features.ndim != 4 or not torch.is_floating_point(features) or not torch.isfinite(features).all():  # 按当前条件选择后续控制路径。
            raise ValueError("RPN expects finite floating NCHW features")  # 遇到非法合同立即显式失败。
        hidden = F.relu(self.shared(features))  # 计算并保存当前步骤的中间状态。
        logits = self.objectness(hidden).permute(0, 2, 3, 1).reshape(features.shape[0], -1)  # 计算并保存当前步骤的中间状态。
        deltas = self.box_deltas(hidden).reshape(features.shape[0], self.anchors_per_location, 4,  # 计算并保存当前步骤的中间状态。
                                                       features.shape[2], features.shape[3])  # 执行当前语句以推进本节示例。
        deltas = deltas.permute(0, 3, 4, 1, 2).reshape(features.shape[0], -1, 4)  # 计算并保存当前步骤的中间状态。
        return logits, deltas  # 返回当前分支计算出的结果。

def rpn_losses(logits, deltas, anchors, targets, generator, anchor_valid=None):  # 定义本节可复用的核心函数。
    if (logits.ndim != 2 or deltas.ndim != 3 or logits.shape[:2] != deltas.shape[:2]  # 按当前条件选择后续控制路径。
            or deltas.shape[-1] != 4 or logits.shape[1] != anchors.shape[0]):  # 计算并保存当前步骤的中间状态。
        raise ValueError("RPN prediction/anchor shape mismatch")  # 遇到非法合同立即显式失败。
    if not torch.is_floating_point(logits) or not torch.is_floating_point(deltas):  # 按当前条件选择后续控制路径。
        raise ValueError("RPN predictions must be floating point")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(logits).all() or not torch.isfinite(deltas).all():  # 按当前条件选择后续控制路径。
        raise ValueError("RPN predictions must be finite")  # 遇到非法合同立即显式失败。
    validate_boxes(anchors)  # 执行当前语句以推进本节示例。
    if len(targets) != logits.shape[0]:  # 按当前条件选择后续控制路径。
        raise ValueError("one RPN target per image is required")  # 遇到非法合同立即显式失败。
    cls_losses, reg_losses = [], []  # 计算并保存当前步骤的中间状态。
    if anchor_valid is None:  # 按当前条件选择后续控制路径。
        anchor_valid = torch.ones_like(logits, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    if anchor_valid.shape != logits.shape or anchor_valid.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise ValueError("anchor_valid must be bool and align with objectness")  # 遇到非法合同立即显式失败。
    for batch_index, target in enumerate(targets):  # 遍历输入元素以累积或检查结果。
        if set(target) != {"boxes", "labels"}:  # 按当前条件选择后续控制路径。
            raise ValueError("RPN target must contain boxes and labels")  # 遇到非法合同立即显式失败。
        labels, matched = match_anchors(anchors, target["boxes"])  # 计算并保存当前步骤的中间状态。
        labels[~anchor_valid[batch_index]] = -1  # 计算并保存当前步骤的中间状态。
        # 全 padding 且空 GT 是合法占位样本，返回与预测图相连的可导零。
        if not bool(anchor_valid[batch_index].any()):  # 按当前条件选择后续控制路径。
            if target["boxes"].numel():  # 按当前条件选择后续控制路径。
                raise ValueError("fully padded image cannot carry ground truth")  # 遇到非法合同立即显式失败。
            cls_losses.append(logits[batch_index].sum() * 0)  # 执行当前语句以推进本节示例。
            reg_losses.append(deltas[batch_index].sum() * 0)  # 执行当前语句以推进本节示例。
            continue  # 调整当前循环或占位控制流。
        sampled, positives = balanced_sample(labels, RPN_BATCH_SIZE47, RPN_POSITIVE_FRACTION47, generator)  # 计算并保存当前步骤的中间状态。
        if sampled.numel():  # 按当前条件选择后续控制路径。
            cls_sum = F.binary_cross_entropy_with_logits(  # 计算并保存当前步骤的中间状态。
                logits[batch_index, sampled], labels[sampled].float(), reduction="sum")  # 计算并保存当前步骤的中间状态。
            cls_losses.append(cls_sum / sampled.numel())  # 执行当前语句以推进本节示例。
        else:  # 处理前置条件不成立的分支。
            cls_losses.append(logits[batch_index].sum() * 0)  # 执行当前语句以推进本节示例。
        if positives.numel():  # 按当前条件选择后续控制路径。
            wanted = box_coder47.encode(anchors[positives], target["boxes"][matched[positives]])  # 计算并保存当前步骤的中间状态。
            reg_sum = F.smooth_l1_loss(deltas[batch_index, positives], wanted,  # 计算并保存当前步骤的中间状态。
                                       beta=RPN_SMOOTH_L1_BETA47, reduction="sum")  # 计算并保存当前步骤的中间状态。
            reg_losses.append(reg_sum / (positives.numel() * 4))  # 执行当前语句以推进本节示例。
        else:  # 处理前置条件不成立的分支。
            reg_losses.append(deltas[batch_index].sum() * 0)  # 执行当前语句以推进本节示例。
    return torch.stack(cls_losses).mean(), torch.stack(reg_losses).mean()  # 返回当前分支计算出的结果。

rpn47 = RegionProposalNetwork(16)  # 计算并保存当前步骤的中间状态。
rpn_logits_probe, rpn_deltas_probe = rpn47(torch.randn(2, 16, 8, 8))  # 计算并保存当前步骤的中间状态。
assert rpn_logits_probe.shape == (2, 192)  # 用受控断言验证关键不变量。
assert rpn_deltas_probe.shape == (2, 192, 4)  # 用受控断言验证关键不变量。
assert torch.isfinite(rpn_logits_probe).all() and torch.isfinite(rpn_deltas_probe).all()  # 用受控断言验证关键不变量。

all_invalid_logits47 = torch.zeros(1, anchors47.shape[0], requires_grad=True)  # 计算并保存当前步骤的中间状态。
all_invalid_deltas47 = torch.zeros(1, anchors47.shape[0], 4, requires_grad=True)  # 计算并保存当前步骤的中间状态。
empty_rpn_target47 = [{"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)}]  # 计算并保存当前步骤的中间状态。
zero_rpn_cls47, zero_rpn_reg47 = rpn_losses(  # 计算并保存当前步骤的中间状态。
    all_invalid_logits47, all_invalid_deltas47, anchors47, empty_rpn_target47,  # 执行当前语句以推进本节示例。
    torch.Generator().manual_seed(47), torch.zeros_like(all_invalid_logits47, dtype=torch.bool))  # 计算并保存当前步骤的中间状态。
assert torch.equal(zero_rpn_cls47, torch.tensor(0.0)) and torch.equal(zero_rpn_reg47, torch.tensor(0.0))  # 用受控断言验证关键不变量。
assert torch.isfinite(zero_rpn_cls47 + zero_rpn_reg47)  # 用受控断言验证关键不变量。
(zero_rpn_cls47 + zero_rpn_reg47).backward()  # 执行当前语句以推进本节示例。
assert all_invalid_logits47.grad is not None and torch.count_nonzero(all_invalid_logits47.grad) == 0  # 用受控断言验证关键不变量。


## 6. Proposal：decode、clip、尺寸过滤、top-k 与手写 NMS

NMS 按 score 从高到低保留 box，并删除与当前 box 的 IoU 超过阈值者。顺序很重要：应先 clip/删除退化框，再做 NMS。随机 RPN 可能产生零 proposal，后续 ROI 与推理都必须把空张量当正常输入，而不是假设至少有一个框。


In [ ]:
RPN_PROPOSAL_SCORE47 = 0.0  # 计算并保存当前步骤的中间状态。
RPN_PRE_NMS_TOPK47 = 80  # 计算并保存当前步骤的中间状态。
RPN_POST_NMS47 = 24  # 计算并保存当前步骤的中间状态。
RPN_NMS_IOU47 = 0.7  # 计算并保存当前步骤的中间状态。
RPN_MIN_SIZE47 = 1.0  # 计算并保存当前步骤的中间状态。

def nms(boxes, scores, iou_threshold):  # 定义本节可复用的核心函数。
    validate_boxes(boxes)  # 执行当前语句以推进本节示例。
    if scores.ndim != 1 or scores.shape[0] != boxes.shape[0] or not torch.is_floating_point(scores) or not torch.isfinite(scores).all():  # 按当前条件选择后续控制路径。
        raise ValueError("scores must align and be finite floating point")  # 遇到非法合同立即显式失败。
    if not math.isfinite(iou_threshold) or not 0 <= iou_threshold <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("invalid NMS threshold")  # 遇到非法合同立即显式失败。
    if boxes.shape[0] == 0:  # 按当前条件选择后续控制路径。
        return torch.empty(0, dtype=torch.long, device=boxes.device)  # 返回当前分支计算出的结果。
    order = scores.argsort(descending=True)  # 计算并保存当前步骤的中间状态。
    keep = []  # 计算并保存当前步骤的中间状态。
    while order.numel():  # 在终止条件满足前持续推进状态。
        current = order[0]  # 计算并保存当前步骤的中间状态。
        keep.append(current)  # 执行当前语句以推进本节示例。
        if order.numel() == 1:  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
        rest = order[1:]  # 计算并保存当前步骤的中间状态。
        order = rest[box_iou(boxes[current:current+1], boxes[rest]).squeeze(0) <= iou_threshold]  # 计算并保存当前步骤的中间状态。
    return torch.stack(keep)  # 返回当前分支计算出的结果。

def generate_proposals(logits, deltas, anchors, image_size, score_threshold=RPN_PROPOSAL_SCORE47,  # 定义本节可复用的核心函数。
                       topk=RPN_PRE_NMS_TOPK47, post_nms=RPN_POST_NMS47):  # 计算并保存当前步骤的中间状态。
    if logits.ndim != 1 or not torch.is_floating_point(logits) or not torch.isfinite(logits).all():  # 按当前条件选择后续控制路径。
        raise ValueError("proposal logits must be finite floating [A]")  # 遇到非法合同立即显式失败。
    if deltas.shape != anchors.shape or logits.shape[0] != anchors.shape[0]:  # 按当前条件选择后续控制路径。
        raise ValueError("proposal predictions must align with anchors")  # 遇到非法合同立即显式失败。
    if not math.isfinite(score_threshold) or not 0 <= score_threshold <= 1 or topk <= 0 or post_nms < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("proposal filtering recipe is invalid")  # 遇到非法合同立即显式失败。
    boxes = clip_boxes(box_coder47.decode(anchors, deltas), image_size)  # 计算并保存当前步骤的中间状态。
    scores = logits.sigmoid()  # 计算并保存当前步骤的中间状态。
    wh = boxes[:, 2:] - boxes[:, :2]  # 计算并保存当前步骤的中间状态。
    valid = (wh[:, 0] >= RPN_MIN_SIZE47) & (wh[:, 1] >= RPN_MIN_SIZE47) & (scores >= score_threshold)  # 计算并保存当前步骤的中间状态。
    boxes, scores = boxes[valid], scores[valid]  # 计算并保存当前步骤的中间状态。
    if boxes.shape[0] == 0:  # 按当前条件选择后续控制路径。
        return boxes, scores  # 返回当前分支计算出的结果。
    top = scores.argsort(descending=True)[:topk]  # 计算并保存当前步骤的中间状态。
    boxes, scores = boxes[top], scores[top]  # 计算并保存当前步骤的中间状态。
    keep = nms(boxes, scores, RPN_NMS_IOU47)[:post_nms]  # 计算并保存当前步骤的中间状态。
    return boxes[keep], scores[keep]  # 返回当前分支计算出的结果。

nms_boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.], [20., 20., 30., 30.]])  # 计算并保存当前步骤的中间状态。
nms_scores = torch.tensor([0.9, 0.8, 0.7])  # 计算并保存当前步骤的中间状态。
assert torch.equal(nms(nms_boxes, nms_scores, 0.5), torch.tensor([0, 2]))  # 用受控断言验证关键不变量。
assert nms(torch.empty(0, 4), torch.empty(0), 0.5).numel() == 0  # 用受控断言验证关键不变量。
none_boxes, none_scores = generate_proposals(torch.full((192,), -20.), torch.zeros(192, 4),  # 计算并保存当前步骤的中间状态。
                                              anchors47, (32, 32), score_threshold=0.99)  # 计算并保存当前步骤的中间状态。
assert none_boxes.shape == (0, 4) and none_scores.shape == (0,)  # 用受控断言验证关键不变量。


## 7. `IntegerROIPool`：它不是 ROI Align

本实现把 image 坐标乘 `feature/image` scale，左上取 floor、右下取 ceil，裁剪整数 feature 区域后做 `adaptive_max_pool2d`。这相当于经典 ROI pooling 的简化版，会有量化误差；ROI Align 则在浮点采样点做双线性插值并避免两次取整。为了不误导，类名和接口都明确写出 `Integer`。


In [ ]:
class IntegerROIPool(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, output_size=(3, 3)):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.output_size = tuple(output_size)  # 计算并保存当前步骤的中间状态。

    def forward(self, features, rois, image_size):  # 定义本节可复用的核心函数。
        if features.ndim != 4 or rois.ndim != 2 or rois.shape[1] != 5:  # 按当前条件选择后续控制路径。
            raise ValueError("features NCHW and rois [R,batch,x1,y1,x2,y2] required")  # 遇到非法合同立即显式失败。
        if not torch.isfinite(rois).all():  # 按当前条件选择后续控制路径。
            raise ValueError("rois must be finite")  # 遇到非法合同立即显式失败。
        if rois.shape[0] == 0:  # 按当前条件选择后续控制路径。
            return features.new_empty((0, features.shape[1], *self.output_size))  # 返回当前分支计算出的结果。
        validate_boxes(rois[:, 1:], image_size)  # 执行当前语句以推进本节示例。
        hi, wi = image_size; hf, wf = features.shape[-2:]  # 计算并保存当前步骤的中间状态。
        pooled = []  # 计算并保存当前步骤的中间状态。
        for roi in rois:  # 遍历输入元素以累积或检查结果。
            batch_index = int(roi[0].item())  # 计算并保存当前步骤的中间状态。
            if batch_index < 0 or batch_index >= features.shape[0] or float(roi[0]) != batch_index:  # 按当前条件选择后续控制路径。
                raise ValueError("ROI batch index is invalid")  # 遇到非法合同立即显式失败。
            x1 = max(0, min(wf - 1, math.floor(float(roi[1]) * wf / wi)))  # 计算并保存当前步骤的中间状态。
            y1 = max(0, min(hf - 1, math.floor(float(roi[2]) * hf / hi)))  # 计算并保存当前步骤的中间状态。
            x2 = max(x1 + 1, min(wf, math.ceil(float(roi[3]) * wf / wi)))  # 计算并保存当前步骤的中间状态。
            y2 = max(y1 + 1, min(hf, math.ceil(float(roi[4]) * hf / hi)))  # 计算并保存当前步骤的中间状态。
            pooled.append(F.adaptive_max_pool2d(features[batch_index:batch_index+1, :, y1:y2, x1:x2], self.output_size)[0])  # 执行当前语句以推进本节示例。
        return torch.stack(pooled)  # 返回当前分支计算出的结果。

roi_pool47 = IntegerROIPool((2, 2))  # 计算并保存当前步骤的中间状态。
grid_feature = torch.arange(64, dtype=torch.float32).reshape(1, 1, 8, 8)  # 计算并保存当前步骤的中间状态。
grid_roi = torch.tensor([[0., 8., 8., 16., 16.]])  # 计算并保存当前步骤的中间状态。
grid_pooled = roi_pool47(grid_feature, grid_roi, (32, 32))  # 计算并保存当前步骤的中间状态。
assert torch.equal(grid_pooled[0, 0], torch.tensor([[18., 19.], [26., 27.]]))  # 用受控断言验证关键不变量。
assert roi_pool47(torch.randn(2, 4, 8, 8), torch.empty(0, 5), (32, 32)).shape == (0, 4, 2, 2)  # 用受控断言验证关键不变量。


## 8. 二阶段 head 与整网 forward

每个 proposal 的 pooled feature 进入两层 MLP，输出 `K+1` 类和一组 class-agnostic delta。训练时把 GT box 追加到 RPN proposals，确保早期至少存在正 ROI；这只是训练采样策略，推理不能偷看 GT。空 GT 图像仍可贡献 background 分类损失。


In [ ]:
class TwoStageHead(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=16, pool_size=3, num_classes=2, hidden=48):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.num_classes = int(num_classes)  # 计算并保存当前步骤的中间状态。
        self.fc1 = nn.Linear(in_channels * pool_size * pool_size, hidden)  # 计算并保存当前步骤的中间状态。
        self.fc2 = nn.Linear(hidden, hidden)  # 计算并保存当前步骤的中间状态。
        self.classifier = nn.Linear(hidden, num_classes + 1)  # 计算并保存当前步骤的中间状态。
        self.regressor = nn.Linear(hidden, 4)  # 计算并保存当前步骤的中间状态。

    def forward(self, pooled):  # 定义本节可复用的核心函数。
        if pooled.ndim != 4:  # 按当前条件选择后续控制路径。
            raise ValueError("pooled features must be RCHW")  # 遇到非法合同立即显式失败。
        if pooled.shape[0] == 0:  # 按当前条件选择后续控制路径。
            return pooled.new_empty((0, self.num_classes + 1)), pooled.new_empty((0, 4))  # 返回当前分支计算出的结果。
        hidden = F.relu(self.fc1(pooled.flatten(1)))  # 计算并保存当前步骤的中间状态。
        hidden = F.relu(self.fc2(hidden))  # 计算并保存当前步骤的中间状态。
        return self.classifier(hidden), self.regressor(hidden)  # 返回当前分支计算出的结果。

def validate_targets47(targets, batch_size, image_size, num_classes):  # 定义本节可复用的核心函数。
    if len(targets) != batch_size:  # 按当前条件选择后续控制路径。
        raise ValueError("one target dictionary per image is required")  # 遇到非法合同立即显式失败。
    for target in targets:  # 遍历输入元素以累积或检查结果。
        if set(target) != {"boxes", "labels"}:  # 按当前条件选择后续控制路径。
            raise ValueError("target must contain exactly boxes and labels")  # 遇到非法合同立即显式失败。
        validate_boxes(target["boxes"], image_size)  # 执行当前语句以推进本节示例。
        labels = target["labels"]  # 计算并保存当前步骤的中间状态。
        if labels.ndim != 1 or labels.dtype != torch.long or labels.shape[0] != target["boxes"].shape[0]:  # 按当前条件选择后续控制路径。
            raise ValueError("labels must be int64 and align with boxes")  # 遇到非法合同立即显式失败。
        if labels.numel() and ((labels < 1).any() or (labels > num_classes).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("foreground labels must be in [1,num_classes]")  # 遇到非法合同立即显式失败。

class TinyFasterRCNN(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, num_classes=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.num_classes = int(num_classes)  # 计算并保存当前步骤的中间状态。
        self.backbone = TinyDetectionBackbone(1, 16)  # 计算并保存当前步骤的中间状态。
        self.anchor_generator = AnchorGenerator(8.0, (0.5, 1.0, 2.0))  # 计算并保存当前步骤的中间状态。
        self.rpn = RegionProposalNetwork(16, 3)  # 计算并保存当前步骤的中间状态。
        self.roi_pool = IntegerROIPool((3, 3))  # 计算并保存当前步骤的中间状态。
        self.head = TwoStageHead(16, 3, num_classes)  # 计算并保存当前步骤的中间状态。

    def forward(self, images, padding_mask, targets=None, score_threshold=0.0):  # 定义本节可复用的核心函数。
        if images.shape[-2:] != (32, 32):  # 按当前条件选择后续控制路径。
            raise ValueError("published teaching model expects 32x32 images")  # 遇到非法合同立即显式失败。
        if targets is not None:  # 按当前条件选择后续控制路径。
            validate_targets47(targets, images.shape[0], (32, 32), self.num_classes)  # 执行当前语句以推进本节示例。
        features, feature_padding = self.backbone(images, padding_mask)  # 计算并保存当前步骤的中间状态。
        fully_invalid = feature_padding.flatten(1).all(1)  # 计算并保存当前步骤的中间状态。
        if targets is not None:  # 按当前条件选择后续控制路径。
            for batch_index, invalid in enumerate(fully_invalid.tolist()):  # 遍历输入元素以累积或检查结果。
                if invalid and targets[batch_index]["boxes"].numel():  # 按当前条件选择后续控制路径。
                    raise ValueError("fully padded image cannot carry ground truth")  # 遇到非法合同立即显式失败。
        anchors = self.anchor_generator(features.shape[-2:], images.shape[-2:]).to(features.device)  # 计算并保存当前步骤的中间状态。
        objectness, rpn_deltas = self.rpn(features)  # 计算并保存当前步骤的中间状态。
        anchor_valid = (~feature_padding).reshape(images.shape[0], -1, 1).expand(-1, -1, 3).reshape(images.shape[0], -1)  # 计算并保存当前步骤的中间状态。
        proposals, proposal_scores = [], []  # 计算并保存当前步骤的中间状态。
        for b in range(images.shape[0]):  # 遍历输入元素以累积或检查结果。
            # Proposal selection/NMS 是离散路径；与标准 two-stage detector 一样在两阶段间 stop-gradient。
            valid = anchor_valid[b]  # 计算并保存当前步骤的中间状态。
            boxes, scores = generate_proposals(objectness[b, valid].detach(), rpn_deltas[b, valid].detach(),  # 计算并保存当前步骤的中间状态。
                                                anchors[valid], (32, 32), score_threshold)  # 执行当前语句以推进本节示例。
            if targets is not None and targets[b]["boxes"].numel():  # 按当前条件选择后续控制路径。
                boxes = torch.cat([boxes, targets[b]["boxes"]], dim=0)  # 计算并保存当前步骤的中间状态。
                scores = torch.cat([scores, torch.ones(targets[b]["boxes"].shape[0], device=scores.device)], dim=0)  # 计算并保存当前步骤的中间状态。
            proposals.append(boxes); proposal_scores.append(scores)  # 执行当前语句以推进本节示例。
        roi_rows = [torch.cat([torch.full((boxes.shape[0], 1), float(b), device=boxes.device), boxes], 1)  # 计算并保存当前步骤的中间状态。
                    for b, boxes in enumerate(proposals) if boxes.shape[0]]  # 遍历输入元素以累积或检查结果。
        rois = torch.cat(roi_rows, 0) if roi_rows else features.new_empty((0, 5))  # 计算并保存当前步骤的中间状态。
        pooled = self.roi_pool(features, rois, (32, 32))  # 计算并保存当前步骤的中间状态。
        class_logits, box_deltas = self.head(pooled)  # 计算并保存当前步骤的中间状态。
        return {"features": features, "feature_padding": feature_padding, "anchors": anchors,  # 返回当前分支计算出的结果。
                "objectness": objectness, "rpn_deltas": rpn_deltas, "anchor_valid": anchor_valid, "proposals": proposals,  # 执行当前语句以推进本节示例。
                "proposal_scores": proposal_scores, "class_logits": class_logits, "box_deltas": box_deltas}  # 执行当前语句以推进本节示例。

detector47 = TinyFasterRCNN(2)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    validate_targets47([{"boxes": torch.tensor([[2., 2., 8., 8.]]), "labels": torch.tensor([0])}], 1, (32, 32), 2)  # 执行当前语句以推进本节示例。
    raise AssertionError("background cannot be supplied as GT")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。


## 9. Detection loss：proposal/target 对齐不能靠数组下标

RPN 与 ROI head 各自重新做 IoU matching。ROI 的 background label 为 0；正 proposal 的回归目标由其 matched GT 编码。若一张图没有 ROI，则跳过该图的 ROI loss，同时保留图中其它 batch 成员的监督。所有 loss 都保持为 tensor，不能用 `.item()` 后再拼回计算图。


In [ ]:
ROI_POSITIVE_IOU47 = 0.4  # 计算并保存当前步骤的中间状态。
ROI_SMOOTH_L1_BETA47 = 1 / 9  # 计算并保存当前步骤的中间状态。
ROI_SAMPLER47 = "all-post-nms-proposals-plus-gt"  # 计算并保存当前步骤的中间状态。

def detection_losses47(outputs, targets, generator):  # 定义本节可复用的核心函数。
    rpn_cls, rpn_reg = rpn_losses(outputs["objectness"], outputs["rpn_deltas"], outputs["anchors"], targets,  # 计算并保存当前步骤的中间状态。
                                  generator, outputs["anchor_valid"])  # 执行当前语句以推进本节示例。
    roi_cls_losses, roi_reg_losses = [], []  # 计算并保存当前步骤的中间状态。
    offset = 0  # 计算并保存当前步骤的中间状态。
    for boxes, target in zip(outputs["proposals"], targets):  # 遍历输入元素以累积或检查结果。
        count = boxes.shape[0]  # 计算并保存当前步骤的中间状态。
        logits = outputs["class_logits"][offset:offset+count]  # 计算并保存当前步骤的中间状态。
        deltas = outputs["box_deltas"][offset:offset+count]  # 计算并保存当前步骤的中间状态。
        offset += count  # 计算并保存当前步骤的中间状态。
        if count == 0:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        if target["boxes"].shape[0] == 0:  # 按当前条件选择后续控制路径。
            labels = torch.zeros(count, dtype=torch.long, device=logits.device)  # 计算并保存当前步骤的中间状态。
            matched = torch.zeros(count, dtype=torch.long, device=logits.device)  # 计算并保存当前步骤的中间状态。
            positive = torch.zeros(count, dtype=torch.bool, device=logits.device)  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            ious = box_iou(boxes, target["boxes"])  # 计算并保存当前步骤的中间状态。
            best_iou, matched = ious.max(1)  # 计算并保存当前步骤的中间状态。
            positive = best_iou >= ROI_POSITIVE_IOU47  # 计算并保存当前步骤的中间状态。
            best_proposal = ious.argmax(0)  # 计算并保存当前步骤的中间状态。
            positive[best_proposal] = True  # 计算并保存当前步骤的中间状态。
            matched[best_proposal] = torch.arange(target["boxes"].shape[0])  # 计算并保存当前步骤的中间状态。
            labels = torch.zeros(count, dtype=torch.long, device=logits.device)  # 计算并保存当前步骤的中间状态。
            labels[positive] = target["labels"][matched[positive]]  # 计算并保存当前步骤的中间状态。
        cls_sum = F.cross_entropy(logits, labels, reduction="sum")  # 计算并保存当前步骤的中间状态。
        roi_cls_losses.append(cls_sum / count)  # 执行当前语句以推进本节示例。
        if positive.any():  # 按当前条件选择后续控制路径。
            wanted = box_coder47.encode(boxes[positive], target["boxes"][matched[positive]])  # 计算并保存当前步骤的中间状态。
            reg_sum = F.smooth_l1_loss(deltas[positive], wanted, beta=ROI_SMOOTH_L1_BETA47, reduction="sum")  # 计算并保存当前步骤的中间状态。
            roi_reg_losses.append(reg_sum / (int(positive.sum()) * 4))  # 执行当前语句以推进本节示例。
        else:  # 处理前置条件不成立的分支。
            roi_reg_losses.append(deltas.sum() * 0)  # 执行当前语句以推进本节示例。
    if offset != outputs["class_logits"].shape[0]:  # 按当前条件选择后续控制路径。
        raise RuntimeError("proposal/head offset mismatch")  # 遇到非法合同立即显式失败。
    zero = outputs["objectness"].sum() * 0  # 计算并保存当前步骤的中间状态。
    roi_cls = torch.stack(roi_cls_losses).mean() if roi_cls_losses else zero  # 计算并保存当前步骤的中间状态。
    roi_reg = torch.stack(roi_reg_losses).mean() if roi_reg_losses else zero  # 计算并保存当前步骤的中间状态。
    return {"rpn_objectness": rpn_cls, "rpn_box": rpn_reg, "roi_class": roi_cls, "roi_box": roi_reg}  # 返回当前分支计算出的结果。

def make_detection_batch47():  # 定义本节可复用的核心函数。
    images = torch.zeros(3, 1, 32, 32)  # 计算并保存当前步骤的中间状态。
    images[0, :, 6:14, 5:13] = 1.0  # 计算并保存当前步骤的中间状态。
    images[1, :, 17:27, 18:28] = 0.8  # 计算并保存当前步骤的中间状态。
    images[2] = 0.03  # 计算并保存当前步骤的中间状态。
    padding = torch.zeros(3, 32, 32, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    padding[1, 30:, :] = True; padding[1, :, 30:] = True  # 计算并保存当前步骤的中间状态。
    targets = [  # 计算并保存当前步骤的中间状态。
        {"boxes": torch.tensor([[5., 6., 13., 14.]]), "labels": torch.tensor([1])},  # 执行当前语句以推进本节示例。
        {"boxes": torch.tensor([[18., 17., 28., 27.]]), "labels": torch.tensor([2])},  # 执行当前语句以推进本节示例。
        {"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)},  # 计算并保存当前步骤的中间状态。
    ]  # 执行当前语句以推进本节示例。
    return images, padding, targets  # 返回当前分支计算出的结果。

train_images47, train_padding47, train_targets47 = make_detection_batch47()  # 计算并保存当前步骤的中间状态。
probe_outputs47 = detector47(train_images47, train_padding47, train_targets47)  # 计算并保存当前步骤的中间状态。
assert not bool(probe_outputs47["anchor_valid"][1].all())  # 用受控断言验证关键不变量。
probe_losses47 = detection_losses47(probe_outputs47, train_targets47, torch.Generator().manual_seed(SEED + 1))  # 计算并保存当前步骤的中间状态。
probe_total47 = sum(probe_losses47.values())  # 计算并保存当前步骤的中间状态。
probe_total47.backward()  # 执行当前语句以推进本节示例。
assert all(torch.isfinite(value) for value in probe_losses47.values())  # 用受控断言验证关键不变量。
assert detector47.backbone.conv1.weight.grad is not None  # 用受控断言验证关键不变量。
assert detector47.rpn.objectness.weight.grad is not None  # 用受控断言验证关键不变量。
assert detector47.head.classifier.weight.grad is not None  # 用受控断言验证关键不变量。
assert float(detector47.head.classifier.weight.grad.abs().sum()) > 0  # 用受控断言验证关键不变量。

# 全 padding + 空 GT 是合法占位样本，所有 loss 为可导零；带 GT 则直接拒绝。
padding_only_model47 = TinyFasterRCNN(2)  # 计算并保存当前步骤的中间状态。
padding_only_image47 = torch.zeros(1, 1, 32, 32)  # 计算并保存当前步骤的中间状态。
padding_only_mask47 = torch.ones(1, 32, 32, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
padding_only_target47 = [{"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)}]  # 计算并保存当前步骤的中间状态。
padding_only_output47 = padding_only_model47(padding_only_image47, padding_only_mask47, padding_only_target47)  # 计算并保存当前步骤的中间状态。
padding_only_losses47 = detection_losses47(  # 计算并保存当前步骤的中间状态。
    padding_only_output47, padding_only_target47, torch.Generator().manual_seed(4701))  # 执行当前语句以推进本节示例。
padding_only_total47 = sum(padding_only_losses47.values())  # 计算并保存当前步骤的中间状态。
assert torch.equal(padding_only_total47, torch.tensor(0.0)) and torch.isfinite(padding_only_total47)  # 用受控断言验证关键不变量。
padding_only_total47.backward()  # 执行当前语句以推进本节示例。
assert padding_only_model47.rpn.objectness.weight.grad is not None  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    padding_only_model47(padding_only_image47, padding_only_mask47,  # 执行当前语句以推进本节示例。
                         [{"boxes": torch.tensor([[2., 2., 8., 8.]]), "labels": torch.tensor([1])}])  # 执行当前语句以推进本节示例。
    raise AssertionError("fully padded image with GT must fail")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "fully padded" in str(exc)  # 用受控断言验证关键不变量。


## 10. 受控微型训练与推理

我们在固定的三张图上短暂优化，观察联合 loss 是否下降；GT 被追加到训练 proposals，因此这是**优化/接线测试**，不是独立测试集。推理时只使用 RPN proposals，class probability 过滤后按类别 NMS。真实检测评估还需要独立 split、AP across IoU thresholds、尺寸分桶和 error analysis。


In [ ]:
TRAIN_STEPS47 = 28  # 计算并保存当前步骤的中间状态。
detector47 = TinyFasterRCNN(2)  # 计算并保存当前步骤的中间状态。
optimizer47 = torch.optim.Adam(detector47.parameters(), lr=3e-3)  # 计算并保存当前步骤的中间状态。
loss_trace47 = []  # 计算并保存当前步骤的中间状态。
detector47.train()  # 执行当前语句以推进本节示例。
for step in range(TRAIN_STEPS47):  # 遍历输入元素以累积或检查结果。
    optimizer47.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    outputs = detector47(train_images47, train_padding47, train_targets47)  # 计算并保存当前步骤的中间状态。
    parts = detection_losses47(outputs, train_targets47, torch.Generator().manual_seed(SEED + 100 + step))  # 计算并保存当前步骤的中间状态。
    total = sum(parts.values())  # 计算并保存当前步骤的中间状态。
    total.backward()  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(detector47.parameters(), 5.0)  # 执行当前语句以推进本节示例。
    optimizer47.step()  # 执行当前语句以推进本节示例。
    loss_trace47.append(float(total.detach()))  # 执行当前语句以推进本节示例。

assert min(loss_trace47[-5:]) < 0.75 * loss_trace47[0]  # 用受控断言验证关键不变量。
assert all(math.isfinite(v) for v in loss_trace47)  # 用受控断言验证关键不变量。
print({"initial_joint_loss": round(loss_trace47[0], 4), "best_last5": round(min(loss_trace47[-5:]), 4)})  # 执行当前语句以推进本节示例。

ROI_SCORE_THRESHOLD47 = 0.2  # 计算并保存当前步骤的中间状态。
ROI_NMS_IOU47 = 0.5  # 计算并保存当前步骤的中间状态。

def postprocess_detections47(proposals, class_logits, box_deltas, image_size, score_threshold=ROI_SCORE_THRESHOLD47):  # 定义本节可复用的核心函数。
    if proposals.ndim != 2 or proposals.shape[-1] != 4 or not torch.is_floating_point(proposals):  # 按当前条件选择后续控制路径。
        raise ValueError("proposals must be floating [R,4]")  # 遇到非法合同立即显式失败。
    if class_logits.ndim != 2 or class_logits.shape[0] != proposals.shape[0] or class_logits.shape[1] < 2:  # 按当前条件选择后续控制路径。
        raise ValueError("class logits must be [R,K+1]")  # 遇到非法合同立即显式失败。
    if box_deltas.shape != proposals.shape or not torch.is_floating_point(box_deltas):  # 按当前条件选择后续控制路径。
        raise ValueError("box deltas must be floating [R,4]")  # 遇到非法合同立即显式失败。
    if not torch.is_floating_point(class_logits):  # 按当前条件选择后续控制路径。
        raise ValueError("class logits must be floating point")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(proposals).all() or not torch.isfinite(class_logits).all() or not torch.isfinite(box_deltas).all():  # 按当前条件选择后续控制路径。
        raise ValueError("postprocess inputs must be finite")  # 遇到非法合同立即显式失败。
    if proposals.device != class_logits.device or proposals.device != box_deltas.device:  # 按当前条件选择后续控制路径。
        raise ValueError("postprocess tensors must share a device")  # 遇到非法合同立即显式失败。
    if not math.isfinite(score_threshold) or not 0 <= score_threshold <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("score_threshold must be finite in [0,1]")  # 遇到非法合同立即显式失败。
    validate_boxes(proposals, image_size)  # 执行当前语句以推进本节示例。
    if proposals.shape[0] == 0:  # 按当前条件选择后续控制路径。
        return {"boxes": proposals, "scores": proposals.new_empty(0),  # 返回当前分支计算出的结果。
                "labels": torch.empty(0, dtype=torch.long, device=proposals.device)}  # 计算并保存当前步骤的中间状态。
    probabilities = class_logits.softmax(-1)  # 计算并保存当前步骤的中间状态。
    all_boxes, all_scores, all_labels = [], [], []  # 计算并保存当前步骤的中间状态。
    decoded = clip_boxes(box_coder47.decode(proposals, box_deltas), image_size)  # 计算并保存当前步骤的中间状态。
    extent = decoded[:, 2:] - decoded[:, :2]  # 计算并保存当前步骤的中间状态。
    valid_extent = (extent[:, 0] >= RPN_MIN_SIZE47) & (extent[:, 1] >= RPN_MIN_SIZE47)  # 计算并保存当前步骤的中间状态。
    for label in range(1, class_logits.shape[1]):  # 遍历输入元素以累积或检查结果。
        scores = probabilities[:, label]  # 计算并保存当前步骤的中间状态。
        selected = (scores >= score_threshold) & valid_extent  # 计算并保存当前步骤的中间状态。
        if not selected.any():  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        boxes_l, scores_l = decoded[selected], scores[selected]  # 计算并保存当前步骤的中间状态。
        keep = nms(boxes_l, scores_l, ROI_NMS_IOU47)  # 计算并保存当前步骤的中间状态。
        all_boxes.append(boxes_l[keep]); all_scores.append(scores_l[keep])  # 执行当前语句以推进本节示例。
        all_labels.append(torch.full((keep.numel(),), label, dtype=torch.long, device=proposals.device))  # 计算并保存当前步骤的中间状态。
    if not all_boxes:  # 按当前条件选择后续控制路径。
        return {"boxes": proposals.new_empty((0, 4)), "scores": proposals.new_empty(0),  # 返回当前分支计算出的结果。
                "labels": torch.empty(0, dtype=torch.long, device=proposals.device)}  # 计算并保存当前步骤的中间状态。
    boxes, scores, labels = torch.cat(all_boxes), torch.cat(all_scores), torch.cat(all_labels)  # 计算并保存当前步骤的中间状态。
    order = scores.argsort(descending=True)  # 计算并保存当前步骤的中间状态。
    return {"boxes": boxes[order], "scores": scores[order], "labels": labels[order]}  # 返回当前分支计算出的结果。

empty_detection47 = postprocess_detections47(torch.empty(0, 4), torch.empty(0, 3), torch.empty(0, 4), (32, 32))  # 计算并保存当前步骤的中间状态。
assert empty_detection47["boxes"].shape == (0, 4)  # 用受控断言验证关键不变量。
assert empty_detection47["labels"].dtype == torch.long  # 用受控断言验证关键不变量。

bad_postprocess47 = [  # 计算并保存当前步骤的中间状态。
    (torch.tensor([[0., 0., 5., 5.]]), torch.tensor([[0., float("nan"), 0.]]), torch.zeros(1, 4), 0.0),  # 执行当前语句以推进本节示例。
    (torch.tensor([[0., 0., 5., 5.]]), torch.zeros(1, 3), torch.full((1, 4), float("inf")), 0.0),  # 执行当前语句以推进本节示例。
    (torch.tensor([[0., 0., 5., 5.]]), torch.zeros(1, 3), torch.zeros(1, 4), float("nan")),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
for bad_boxes47, bad_logits47, bad_deltas47, bad_threshold47 in bad_postprocess47:  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        postprocess_detections47(bad_boxes47, bad_logits47, bad_deltas47, (32, 32), bad_threshold47)  # 执行当前语句以推进本节示例。
        raise AssertionError("nonfinite detection input must fail")  # 遇到非法合同立即显式失败。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        assert "finite" in str(exc)  # 用受控断言验证关键不变量。

detector47.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    inference_outputs47 = detector47(train_images47[:2], train_padding47[:2], targets=None, score_threshold=0.0)  # 计算并保存当前步骤的中间状态。
    first_count47 = inference_outputs47["proposals"][0].shape[0]  # 计算并保存当前步骤的中间状态。
    first_detection47 = postprocess_detections47(  # 计算并保存当前步骤的中间状态。
        inference_outputs47["proposals"][0], inference_outputs47["class_logits"][:first_count47],  # 执行当前语句以推进本节示例。
        inference_outputs47["box_deltas"][:first_count47], (32, 32), 0.05)  # 执行当前语句以推进本节示例。
assert first_detection47["boxes"].ndim == 2 and first_detection47["boxes"].shape[-1] == 4  # 用受控断言验证关键不变量。
assert torch.isfinite(first_detection47["scores"]).all()  # 用受控断言验证关键不变量。


## 11. 发布边界：坐标、anchor、采样、loss 分母和推理阈值都要绑定

检测模型的 state dict 相同，并不意味着系统语义相同：anchor 顺序与 ratio 定义、`xyxy` 是否 inclusive、padding mask、IoU 阈值、正负采样、loss 分母、top-k/NMS、类别起点任一变化都会破坏输出。本制品把这些 recipe 和受控数据快照一并纳入 package digest；canonical state digest 逐参数绑定 key/dtype/shape/bytes。

外部只读 publisher registry 保存发布时批准的整体 digest。loader 还逐项交叉验证 recipe，并返回携带不可变 `preprocess/labels/coordinates/anchors/training/inference` 的 `PublishedDetector`，避免调用方拿裸模型猜语义。攻击者即使替换 head、改 label map并重算全部内部 hash，仍无法改变 registry 中的期望值。


In [ ]:
def state_digest47(state):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(key.encode()); digest.update(str(tensor.dtype).encode())  # 执行当前语句以推进本节示例。
        digest.update(json.dumps(list(tensor.shape)).encode()); digest.update(tensor.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

def data_digest47(images, padding, targets):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for tensor in [images, padding] + [v for target in targets for v in (target["boxes"], target["labels"])]:  # 遍历输入元素以累积或检查结果。
        value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(str(value.dtype).encode()); digest.update(json.dumps(list(value.shape)).encode())  # 执行当前语句以推进本节示例。
        digest.update(value.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

def package_digest47(package):  # 定义本节可复用的核心函数。
    payload = {k: package[k] for k in sorted(package) if k != "package_digest"}  # 计算并保存当前步骤的中间状态。
    return sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()  # 返回当前分支计算出的结果。

def deep_freeze47(value):  # 定义本节可复用的核心函数。
    if isinstance(value, dict): return MappingProxyType({k: deep_freeze47(v) for k, v in value.items()})  # 按当前条件选择后续控制路径。
    if isinstance(value, list): return tuple(deep_freeze47(v) for v in value)  # 按当前条件选择后续控制路径。
    return value  # 返回当前分支计算出的结果。

CONFIG47 = {"image_size": [32, 32], "in_channels": 1, "backbone_channels": 16, "num_classes": 2,  # 计算并保存当前步骤的中间状态。
            "roi_pool": [3, 3], "roi_method": "integer-floor-ceil-crop-adaptive-max-not-roi-align"}  # 执行当前语句以推进本节示例。
SPLIT47 = {"kind": "controlled-three-image-optimization", "seed": SEED,  # 计算并保存当前步骤的中间状态。
           "independent_test": False, "claim": "wiring-and-optimization-only"}  # 执行当前语句以推进本节示例。
PREPROCESS47 = {"range": [0.0, 1.0], "dtype": "float32", "layout": "NCHW",  # 计算并保存当前步骤的中间状态。
                "padding_mask": "bool-True-is-invalid-zero-before-conv", "fully_padded_empty_gt": "zero-loss"}  # 执行当前语句以推进本节示例。
COORDINATES47 = {"format": "continuous-xyxy-exclusive-upper-bound", "clip": "[0,W]x[0,H]",  # 计算并保存当前步骤的中间状态。
                 "width_height": "x2-x1,y2-y1", "roi_scale": "floor-left-top-ceil-right-bottom"}  # 执行当前语句以推进本节示例。
ANCHORS_RECIPE47 = {"flatten_order": "y-x-ratio", "center": "(index+0.5)*image/feature",  # 计算并保存当前步骤的中间状态。
                    "size": 8.0, "aspect_ratios": [0.5, 1.0, 2.0], "ratio_definition": "width/height",  # 执行当前语句以推进本节示例。
                    "border_policy": "allow-straddling-then-clip-proposals"}  # 执行当前语句以推进本节示例。
TRAINING_RECIPE47 = {  # 计算并保存当前步骤的中间状态。
    "optimizer": "Adam", "lr": 3e-3, "steps": TRAIN_STEPS47,  # 执行当前语句以推进本节示例。
    "rpn": {"positive_iou": RPN_POSITIVE_IOU47, "negative_iou": RPN_NEGATIVE_IOU47,  # 执行当前语句以推进本节示例。
            "batch_size": RPN_BATCH_SIZE47, "positive_fraction": RPN_POSITIVE_FRACTION47,  # 执行当前语句以推进本节示例。
            "smooth_l1_beta": RPN_SMOOTH_L1_BETA47,  # 执行当前语句以推进本节示例。
            "classification_denominator": "sampled-anchor-count-or-differentiable-zero",  # 执行当前语句以推进本节示例。
            "regression_denominator": "positive-anchor-coordinate-count"},  # 执行当前语句以推进本节示例。
    "roi": {"positive_iou": ROI_POSITIVE_IOU47, "negative_iou": "below-positive",  # 执行当前语句以推进本节示例。
            "ignore": None, "sampler": ROI_SAMPLER47, "smooth_l1_beta": ROI_SMOOTH_L1_BETA47,  # 执行当前语句以推进本节示例。
            "classification_denominator": "proposal-count-per-image-then-image-mean",  # 执行当前语句以推进本节示例。
            "regression_denominator": "positive-roi-coordinate-count-then-image-mean"},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
INFERENCE_RECIPE47 = {  # 计算并保存当前步骤的中间状态。
    "rpn_score": RPN_PROPOSAL_SCORE47, "rpn_pre_nms_topk": RPN_PRE_NMS_TOPK47,  # 执行当前语句以推进本节示例。
    "rpn_post_nms": RPN_POST_NMS47, "rpn_nms_iou": RPN_NMS_IOU47, "min_box_size": RPN_MIN_SIZE47,  # 执行当前语句以推进本节示例。
    "proposal_stop_gradient": True, "roi_score_default": ROI_SCORE_THRESHOLD47,  # 执行当前语句以推进本节示例。
    "roi_per_class_nms_iou": ROI_NMS_IOU47, "box_regression": "class-agnostic",  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
LABELS47 = {"0": "background", "1": "class-one-square", "2": "class-two-square"}  # 计算并保存当前步骤的中间状态。

state47 = {k: v.detach().cpu().clone() for k, v in detector47.state_dict().items()}  # 计算并保存当前步骤的中间状态。
buffer47 = io.BytesIO(); torch.save(state47, buffer47)  # 计算并保存当前步骤的中间状态。
artifact47 = {  # 计算并保存当前步骤的中间状态。
    "artifact_id": "tiny-faster-rcnn-squares-v1", "config": CONFIG47,  # 执行当前语句以推进本节示例。
    "state_hex": buffer47.getvalue().hex(), "state_digest": state_digest47(state47),  # 执行当前语句以推进本节示例。
    "data_digest": data_digest47(train_images47, train_padding47, train_targets47),  # 执行当前语句以推进本节示例。
    "split": SPLIT47, "preprocess": PREPROCESS47, "coordinates": COORDINATES47,  # 执行当前语句以推进本节示例。
    "anchors": ANCHORS_RECIPE47, "training_recipe": TRAINING_RECIPE47,  # 执行当前语句以推进本节示例。
    "inference_recipe": INFERENCE_RECIPE47, "labels": LABELS47,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact47["package_digest"] = package_digest47(artifact47)  # 计算并保存当前步骤的中间状态。
PUBLISHER_REGISTRY47 = MappingProxyType({artifact47["artifact_id"]: artifact47["package_digest"]})  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class PublishedDetector:  # 定义承载本节状态与行为的数据结构。
    model: TinyFasterRCNN  # 执行当前语句以推进本节示例。
    config: object  # 执行当前语句以推进本节示例。
    preprocess: object  # 执行当前语句以推进本节示例。
    labels: object  # 执行当前语句以推进本节示例。
    coordinates: object  # 执行当前语句以推进本节示例。
    anchors: object  # 执行当前语句以推进本节示例。
    training: object  # 执行当前语句以推进本节示例。
    inference: object  # 执行当前语句以推进本节示例。

    def forward(self, images, padding_mask):  # 定义本节可复用的核心函数。
        if images.dtype != torch.float32 or not torch.isfinite(images).all():  # 按当前条件选择后续控制路径。
            raise ValueError("published detector expects finite float32 images")  # 遇到非法合同立即显式失败。
        if images.numel() and (float(images.min()) < 0 or float(images.max()) > 1):  # 按当前条件选择后续控制路径。
            raise ValueError("published detector expects image range [0,1]")  # 遇到非法合同立即显式失败。
        if padding_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("published detector expects bool padding mask")  # 遇到非法合同立即显式失败。
        return self.model(images, padding_mask, targets=None,  # 返回当前分支计算出的结果。
                          score_threshold=float(self.inference["rpn_score"]))  # 计算并保存当前步骤的中间状态。

def load_published_detector47(package):  # 定义本节可复用的核心函数。
    artifact_id = package.get("artifact_id")  # 计算并保存当前步骤的中间状态。
    if artifact_id not in PUBLISHER_REGISTRY47 or package.get("package_digest") != PUBLISHER_REGISTRY47[artifact_id]:  # 按当前条件选择后续控制路径。
        raise ValueError("artifact is not approved by publisher registry")  # 遇到非法合同立即显式失败。
    if package_digest47(package) != package["package_digest"]:  # 按当前条件选择后续控制路径。
        raise ValueError("package digest mismatch")  # 遇到非法合同立即显式失败。
    expected = {"config": CONFIG47, "split": SPLIT47, "preprocess": PREPROCESS47,  # 计算并保存当前步骤的中间状态。
                "coordinates": COORDINATES47, "anchors": ANCHORS_RECIPE47,  # 执行当前语句以推进本节示例。
                "training_recipe": TRAINING_RECIPE47, "inference_recipe": INFERENCE_RECIPE47,  # 执行当前语句以推进本节示例。
                "labels": LABELS47}  # 执行当前语句以推进本节示例。
    for field, wanted in expected.items():  # 遍历输入元素以累积或检查结果。
        if package.get(field) != wanted:  # 按当前条件选择后续控制路径。
            raise ValueError(f"published detector contract mismatch: {field}")  # 遇到非法合同立即显式失败。
    if set(package["labels"]) != {str(i) for i in range(package["config"]["num_classes"] + 1)}:  # 按当前条件选择后续控制路径。
        raise ValueError("detector labels and num_classes disagree")  # 遇到非法合同立即显式失败。
    if package.get("data_digest") != data_digest47(train_images47, train_padding47, train_targets47):  # 按当前条件选择后续控制路径。
        raise ValueError("bound controlled data mismatch")  # 遇到非法合同立即显式失败。
    state = torch.load(io.BytesIO(bytes.fromhex(package["state_hex"])), map_location="cpu", weights_only=True)  # 计算并保存当前步骤的中间状态。
    if state_digest47(state) != package["state_digest"]:  # 按当前条件选择后续控制路径。
        raise ValueError("canonical state digest mismatch")  # 遇到非法合同立即显式失败。
    model = TinyFasterRCNN(num_classes=package["config"]["num_classes"]).eval()  # 计算并保存当前步骤的中间状态。
    model.load_state_dict(state, strict=True)  # 计算并保存当前步骤的中间状态。
    return PublishedDetector(model, deep_freeze47(package["config"]), deep_freeze47(package["preprocess"]),  # 返回当前分支计算出的结果。
                             deep_freeze47(package["labels"]), deep_freeze47(package["coordinates"]),  # 执行当前语句以推进本节示例。
                             deep_freeze47(package["anchors"]), deep_freeze47(package["training_recipe"]),  # 执行当前语句以推进本节示例。
                             deep_freeze47(package["inference_recipe"]))  # 执行当前语句以推进本节示例。

loaded_detector47 = load_published_detector47(deepcopy(artifact47))  # 计算并保存当前步骤的中间状态。
assert isinstance(loaded_detector47.model, TinyFasterRCNN)  # 用受控断言验证关键不变量。
assert loaded_detector47.labels["0"] == "background"  # 用受控断言验证关键不变量。
assert loaded_detector47.inference["proposal_stop_gradient"] is True  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    loaded_detector47.labels["1"] = "mutated"  # 计算并保存当前步骤的中间状态。
    raise AssertionError("published detector labels should be read-only")  # 遇到非法合同立即显式失败。
except TypeError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

forged47 = deepcopy(artifact47)  # 计算并保存当前步骤的中间状态。
forged_state47 = torch.load(io.BytesIO(bytes.fromhex(forged47["state_hex"])), weights_only=True)  # 计算并保存当前步骤的中间状态。
forged_state47["head.classifier.bias"] = forged_state47["head.classifier.bias"].roll(1)  # 计算并保存当前步骤的中间状态。
forged_buffer47 = io.BytesIO(); torch.save(forged_state47, forged_buffer47)  # 计算并保存当前步骤的中间状态。
forged47["state_hex"] = forged_buffer47.getvalue().hex()  # 计算并保存当前步骤的中间状态。
forged47["state_digest"] = state_digest47(forged_state47)  # 计算并保存当前步骤的中间状态。
forged47["labels"] = {"0": "background", "1": "forged", "2": "mapping"}  # 计算并保存当前步骤的中间状态。
forged47["package_digest"] = package_digest47(forged47)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_published_detector47(forged47)  # 执行当前语句以推进本节示例。
    raise AssertionError("fully rehashed forged detector must fail closed")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "publisher registry" in str(exc)  # 用受控断言验证关键不变量。


## 12. 失败模式、复杂度与生产差距

- **空集合**：空 GT、空 proposals、某类别无检测都是正常分支；对空 tensor 直接 `.mean()` 会产生 NaN。
- **坐标与 scale**：本例固定 32×32 和连续 `xyxy`；真实 resize/letterbox 必须保存原尺寸与 scale，推理后映射回原图。
- **ROI 算子**：整数 pooling 有量化误差，不可在文档或模型名中冒充 ROI Align。生产应使用经过数值/梯度验证的高性能算子。
- **padding 感受野**：本例保守清零所有接触 padding 的 feature；真实 FPN 多尺度需要逐层传播 mask。
- **复杂度**：RPN 是 $O(HWA)$；朴素 NMS 最坏 $O(P^2)$；逐 ROI Python 循环只适合教学。
- **发布语义**：`PublishedDetector` 携带只读预处理、坐标、anchor、训练分母与推理阈值；真实 registry 应由签名/KMS 托管。
- **评估**：三张训练图上的 loss 下降不代表泛化。生产需要独立 train/val/test、COCO 风格 AP、类别/尺度分桶、阈值校准和延迟压测。

原始资料：

- [Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks](https://arxiv.org/abs/1506.01497)
- [Fast R-CNN（ROI pooling）](https://arxiv.org/abs/1504.08083)
- [Mask R-CNN（ROI Align）](https://arxiv.org/abs/1703.06870)
